# SmartHome-Bench: 유리문 앞 점프 횟수 검증

이 노트북은 [SmartHome-Bench](https://huggingface.co/datasets/violetcliff/SmartHome-Bench)의 `smartbench_0936` 공개 영상을 사용합니다. [원본 영상](https://www.youtube.com/shorts/G6jUprbgHig)은 로컬에만 저장하고 Git에는 넣지 않습니다. 질문은 **“강아지가 유리문 앞에서 몇 번 점프했어?”**입니다. 영상 설명의 “반복”이라는 표현은 정답 횟수로 쓰지 않습니다.


## 1. 영상 검토 기준

사용자는 영상에서 동작 **8회**를 확인했습니다. 그중 **6회는 뒷발도 바닥에서 떨어진 확실한 점프**, **2회는 뒷발을 땅에 둔 채 앞발만 든 동작**입니다(약 6.2초·7.5초). 이번 질문에서는 앞발만 든 동작도 포함해 **앞발을 들었다가 다시 내리는 한 차례를 1회**로 셉니다. 따라서 비교할 정답은 **8회**입니다. 개별 시작·끝 시각은 0.25초 간격 프레임에서 추정한 잠정값입니다.


In [1]:
import json
from pathlib import Path
base = Path.cwd()
folder = base / "notebooks" if (base / "notebooks").is_dir() else base
review = json.loads((folder / "smarthome-bench-0936-review.json").read_text())
counts = review["user_review"]
print(f"확실한 점프 {counts['confirmed_jumps']}회 + 앞발만 든 동작 {counts['ambiguous_front_paws_only']}회 = 이번 집계 기준 {review['inclusive_question_rule']['expected_episode_count']}회")
print(f"앞발만 든 동작 시각: {counts['ambiguous_times_confirmed_by_user_seconds']}초")

확실한 점프 6회 + 앞발만 든 동작 2회 = 이번 집계 기준 8회
앞발만 든 동작 시각: [6.2, 7.5]초


## 2. 로컬 영상 재생

아래 셀을 로컬 Jupyter에서 실행하면 다운로드한 21.8초 영상을 볼 수 있습니다. 영상 파일은 `data/videos/smartbench_0936.mp4`에 있으며 Git에 포함하지 않습니다.


In [ ]:
from IPython.display import Video, display
video = (folder.parent / "data/videos/smartbench_0936.mp4") if folder.name == "notebooks" else (folder / "data/videos/smartbench_0936.mp4")
if video.exists():
    display(Video(str(video), embed=False))
else:
    print("로컬 영상이 없습니다. Video_url.csv의 smartbench_0936 링크에서 내려받아 data/videos에 저장하세요.")

## 3. 현재 파이프라인과 모델 결과

`gpt-4o-mini`로 기본 2초 간격, 0.5초 간격의 짧은 창, 점프를 직접 묻는 방식을 시험했습니다. 일반 분석은 점프 이벤트를 명시하지 않았고, 앞발 동작까지 포함해 0.25초 프레임으로 전체 영상을 6개 창에서 직접 세자 **8회 대신 19회**를 반환했습니다. 점프가 없는 10.5~14초에도 3회를 만들었습니다. 더 큰 `gpt-4o`도 애매한 구간을 확실한 점프로 분류했습니다. 아래 값은 **검증된 정답이 아니라 실패 사례**입니다.


In [2]:
evaluation = json.loads((folder / "smarthome-bench-0936-eval.json").read_text())
for trial in evaluation["trials"]:
    name = trial["name"]
    if name == "targeted_broad_full_video":
        print(f"{name}: 영상 검토 {trial['human_total']}회, 모델 {trial['model_total']}회")
    elif "events_explicitly_describing_jumps" in trial:
        print(f"{name}: 점프라고 기술된 이벤트 {trial['events_explicitly_describing_jumps']}개 / 전체 {trial['returned_events']}개")
    else:
        print(f"{name}: 영상 검토 확실한 점프 {trial['human_confirmed_jumps']}회, 모델 {trial['model_confirmed_jumps']}회")

current_pipeline_default: 점프라고 기술된 이벤트 0개 / 전체 7개
current_pipeline_dense_windows: 점프라고 기술된 이벤트 0개 / 전체 9개
targeted_prompt_first_5_seconds: 영상 검토 확실한 점프 3회, 모델 2회
targeted_prompt_ambiguous_window: 영상 검토 확실한 점프 0회, 모델 3회
cropped_ambiguous_window: 영상 검토 확실한 점프 0회, 모델 5회
larger_model_ambiguous_window: 영상 검토 확실한 점프 0회, 모델 2회
targeted_broad_full_video: 영상 검토 8회, 모델 19회


## 4. 결론

이번 영상의 집계 기준은 **앞발만 든 동작까지 포함한 8회**입니다. 현재 모델의 전체 영상 결과는 **19회**라서 8회를 맞히지 못했습니다. 모델이 연속 프레임을 서로 다른 행동으로 중복 계산하고, 동작이 없는 구간에도 횟수를 만들었습니다. 뒷발이 뜬 확실한 점프 6회만 세는 엄격한 기준에서도 두 VLM은 안정적이지 않았습니다. 따라서 어느 기준으로도 모델의 정확한 횟수 답변을 발표할 수 없습니다.

현재 집계 라우터의 행동 목록에도 점프가 없어 이 질문은 지원되지 않습니다. 개선하려면 시간상 연결된 자세 변화를 한 동작으로 묶고, 실제 착지나 앞발 내림을 확인한 뒤 다음 동작을 세며, 점프 질문 라우팅을 추가해야 합니다. 이 노트북은 기존 `dog_escape` 영상의 **2회 성공 사례**와 구분해 실패 사례로 발표합니다.
